In [1]:
import pandas as pd
import numpy as np
import pickle
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# LOAD (всё уже в памяти из v5, но на всякий случай)
# ============================================================
history = pd.read_parquet('artifacts/history_interactions.parquet')
games = pd.read_csv('data/game_details.csv')
users = pd.read_csv('data/unique_users.csv')
item_feat_mh = pd.read_parquet('artifacts/item_features_multihot.parquet')

item_pop = pd.read_parquet('artifacts/v2_item_pop.parquet')
item_dev_agg = pd.read_parquet('artifacts/v2_item_dev.parquet')
item_pub_agg = pd.read_parquet('artifacts/v2_item_pub.parquet')
user_stats = pd.read_parquet('artifacts/v2_user_stats.parquet')
user_plat = pd.read_parquet('artifacts/v2_user_plat.parquet')
user_free = pd.read_parquet('artifacts/v2_user_free.parquet')
user_dev = pd.read_parquet('artifacts/v2_user_dev.parquet')
user_pub = pd.read_parquet('artifacts/v2_user_pub.parquet')
dev_per_appid = pd.read_parquet('artifacts/v2_dev_per_appid.parquet')
pub_per_appid = pd.read_parquet('artifacts/v2_pub_per_appid.parquet')

item_dev_te = pd.read_parquet('artifacts/v3_item_dev_te.parquet')
item_pub_te = pd.read_parquet('artifacts/v3_item_pub_te.parquet')
item_genre_te = pd.read_parquet('artifacts/v3_item_genre_te.parquet')
user_dev_te = pd.read_parquet('artifacts/v3_user_dev_te.parquet')
dev_exploded = pd.read_parquet('artifacts/v3_dev_exploded.parquet')

uf_df = pd.read_parquet('artifacts/v4_user_factors.parquet')
if_df = pd.read_parquet('artifacts/v4_item_factors.parquet')
item_factors_normed = np.load('artifacts/v4_item_factors_normed.npy')
user_centroids = np.load('artifacts/v4_user_centroids.npy')
user_centroid_meta = pd.read_parquet('artifacts/v4_user_centroid_meta.parquet')

with open('artifacts/v4_user_top_conf.pkl', 'rb') as f:
    user_top_conf = pickle.load(f)
with open('artifacts/v4_user2idx.pkl', 'rb') as f:
    user2idx = pickle.load(f)
with open('artifacts/v4_item2idx.pkl', 'rb') as f:
    item2idx = pickle.load(f)
with open('artifacts/v5_pca.pkl', 'rb') as f:
    pca = pickle.load(f)

N_FACTORS = 128
N_PCA = 24
uf_cols = [f'uf_{i}' for i in range(N_FACTORS)]
if_cols = [f'if_{i}' for i in range(N_FACTORS)]

mh_genre_cols = [c for c in item_feat_mh.columns if c.startswith('genres_')]
mh_cat_cols = [c for c in item_feat_mh.columns if c.startswith('categories_')]
mh_cols = mh_genre_cols + mh_cat_cols

max_time = 1776010854
user_profile = users[['steamid', 'loccountrycode', 'timecreated']].copy()
user_profile['loccountrycode'] = user_profile['loccountrycode'].fillna('UNKNOWN')
user_profile['account_age_days'] = ((max_time - user_profile['timecreated'].fillna(max_time)) / 86400).clip(lower=0)
country_counts = user_profile['loccountrycode'].value_counts()
user_profile['country_freq'] = user_profile['loccountrycode'].map(country_counts).fillna(0).astype(int)
user_profile = user_profile.drop(columns=['timecreated'])

game_type = games[['appid', 'type']].copy()
game_type['type'] = game_type['type'].fillna('unknown')

# Affinity
hist_mh = history[['steamid', 'appid', 'playtime_forever']].merge(
    item_feat_mh[['appid'] + mh_cols], on='appid', how='inner'
)
hist_mh['log_pt'] = np.log1p(hist_mh['playtime_forever']).astype('float32')

user_game_counts = hist_mh.groupby('steamid').size().reset_index(name='_cnt')
user_aff_count = hist_mh.groupby('steamid')[mh_cols].sum().reset_index()
user_aff_count = user_aff_count.merge(user_game_counts, on='steamid')
for col in mh_cols:
    user_aff_count[f'uaff_{col}'] = (user_aff_count[col] / user_aff_count['_cnt']).astype('float32')
user_aff_count = user_aff_count[['steamid'] + [f'uaff_{c}' for c in mh_cols]]

for col in mh_cols:
    hist_mh[f'w_{col}'] = hist_mh[col].values * hist_mh['log_pt'].values
user_aff_wt = hist_mh.groupby('steamid')[[f'w_{c}' for c in mh_cols]].sum().reset_index()
user_total_wt = hist_mh.groupby('steamid')['log_pt'].sum().reset_index().rename(columns={'log_pt': '_tw'})
user_aff_wt = user_aff_wt.merge(user_total_wt, on='steamid')
for col in mh_cols:
    user_aff_wt[f'uwaff_{col}'] = (user_aff_wt[f'w_{col}'] / user_aff_wt['_tw'].clip(lower=0.001)).astype('float32')
user_aff_wt = user_aff_wt[['steamid'] + [f'uwaff_{c}' for c in mh_cols]]

# Conditional genre stats
hist_with_genres = history[['steamid', 'appid', 'playtime_forever', 'target']].merge(
    item_feat_mh[['appid'] + mh_genre_cols], on='appid', how='inner'
)
genre_records = []
for col in mh_genre_cols:
    subset = hist_with_genres[hist_with_genres[col] == 1][['steamid', 'playtime_forever', 'target']].copy()
    subset['genre'] = col
    genre_records.append(subset)
user_genre_agg = pd.concat(genre_records, ignore_index=True).groupby(['steamid', 'genre']).agg(
    cond_genre_avg_target=('target', 'mean'),
    cond_genre_avg_pt=('playtime_forever', 'mean'),
    cond_genre_n_games=('target', 'count'),
).reset_index()

user_genre_dict = {}
for _, row in user_genre_agg.iterrows():
    key = (row['steamid'], row['genre'])
    user_genre_dict[key] = (row['cond_genre_avg_target'], row['cond_genre_avg_pt'], row['cond_genre_n_games'])

print("All precomputed!")


# ============================================================
# ASSEMBLY v6 — ПОЛНЫЕ 300 кандидатов + лучшие фичи
# ============================================================
def assemble_v6(base_path):
    print(f"\n{'='*60}")
    print(f"Assembling v6: {base_path}")
    print('='*60)
    
    df = pd.read_parquet(base_path)
    df = df.rename(columns={'score': 'als_score', 'rank': 'als_rank'})
    N = len(df)
    
    # [1] ALS SCORE
    print("  [1] ALS score features...")
    df['als_score_log'] = np.log1p(df['als_score'])
    df['als_rank_inv'] = 1.0 / df['als_rank']
    df['als_rank_inv_sqrt'] = 1.0 / np.sqrt(df['als_rank'])
    df['als_rank_norm'] = df['als_rank'] / 300.0
    
    als_g = df.groupby('steamid')['als_score'].agg(['mean', 'std', 'max', 'min']).reset_index()
    als_g.columns = ['steamid', '_am', '_as', '_ax', '_an']
    df = df.merge(als_g, on='steamid')
    df['als_score_zscore'] = (df['als_score'] - df['_am']) / df['_as'].clip(lower=0.001)
    df['als_score_minmax'] = (df['als_score'] - df['_an']) / (df['_ax'] - df['_an']).clip(lower=0.001)
    df.drop(columns=['_am', '_as', '_ax', '_an'], inplace=True)
    
    # [2] ALS FACTORS: PCA(24) + top-8 raw + dot/cosine/norms
    print("  [2] ALS factor products (PCA + raw)...")
    df = df.merge(uf_df, on='steamid', how='left')
    df = df.merge(if_df, on='appid', how='left')
    for c in uf_cols + if_cols:
        df[c] = df[c].fillna(0)
    
    uf_vals = df[uf_cols].values.astype(np.float32)
    if_vals = df[if_cols].values.astype(np.float32)
    
    df['als_dot'] = np.sum(uf_vals * if_vals, axis=1)
    uf_n = np.linalg.norm(uf_vals, axis=1)
    if_n = np.linalg.norm(if_vals, axis=1)
    df['uf_norm'] = uf_n
    df['if_norm'] = if_n
    df['als_cosine'] = df['als_dot'] / (uf_n * if_n + 1e-8)
    
    products = uf_vals * if_vals
    pca_features = pca.transform(products)
    for i in range(N_PCA):
        df[f'pca_{i}'] = pca_features[:, i].astype(np.float32)
    
    product_var = np.var(products, axis=0)
    top8_dims = np.argsort(-product_var)[:8]
    for rank, dim in enumerate(top8_dims):
        df[f'uf_x_if_top{rank}'] = products[:, dim]
    
    df.drop(columns=uf_cols + if_cols, inplace=True)
    
    # [3] ITEM-ITEM SIMILARITY (vectorized)
    print("  [3] Item-item similarity...")
    steamids = df['steamid'].values
    appids = df['appid'].values
    
    user_idx_arr = np.array([user2idx.get(s, -1) for s in steamids], dtype=np.int32)
    item_idx_arr = np.array([item2idx.get(a, -1) for a in appids], dtype=np.int32)
    
    valid_mask = (user_idx_arr >= 0) & (item_idx_arr >= 0)
    centroid_sim = np.zeros(N, dtype=np.float32)
    if valid_mask.any():
        centroid_sim[valid_mask] = np.sum(
            user_centroids[user_idx_arr[valid_mask]] * item_factors_normed[item_idx_arr[valid_mask]], axis=1
        )
    df['centroid_sim'] = centroid_sim
    
    print("    Top-K similarity (batched)...")
    max_sim_arr = np.zeros(N, dtype=np.float32)
    avg_sim_arr = np.zeros(N, dtype=np.float32)
    
    df['_row_idx'] = np.arange(N)
    for sid, indices in df.groupby('steamid')['_row_idx'].apply(list).items():
        if sid not in user_top_conf:
            continue
        top_item_idx = user_top_conf[sid]
        if len(top_item_idx) == 0:
            continue
        cand_appids = appids[indices]
        cand_item_idx = np.array([item2idx.get(a, -1) for a in cand_appids])
        valid = cand_item_idx >= 0
        if not valid.any():
            continue
        sim_matrix = item_factors_normed[top_item_idx] @ item_factors_normed[cand_item_idx[valid]].T
        valid_indices = np.array(indices)[valid]
        max_sim_arr[valid_indices] = sim_matrix.max(axis=0)
        avg_sim_arr[valid_indices] = sim_matrix.mean(axis=0)
    
    df['max_sim_top10'] = max_sim_arr
    df['avg_sim_top10'] = avg_sim_arr
    df['sim_spread'] = max_sim_arr - avg_sim_arr
    df.drop(columns=['_row_idx'], inplace=True)
    
    # [4] USER FEATURES
    print("  [4] User features...")
    df = df.merge(user_profile, on='steamid', how='left')
    df = df.merge(user_stats, on='steamid', how='left')
    df = df.merge(user_plat, on='steamid', how='left')
    df = df.merge(user_free, on='steamid', how='left')
    df = df.merge(user_centroid_meta, on='steamid', how='left')
    
    # [5] ITEM FEATURES
    print("  [5] Item features...")
    item_compact = item_feat_mh[['appid', 'is_free', 'recommendations_log', 'age_years', 'platforms_count']].copy()
    df = df.merge(item_compact, on='appid', how='left')
    df = df.merge(item_pop, on='appid', how='left')
    df = df.merge(item_dev_agg, on='appid', how='left')
    df = df.merge(item_pub_agg, on='appid', how='left')
    df = df.merge(item_dev_te, on='appid', how='left')
    df = df.merge(item_pub_te, on='appid', how='left')
    df = df.merge(item_genre_te, on='appid', how='left')
    df = df.merge(game_type, on='appid', how='left')
    df['type'] = df['type'].fillna('unknown')
    
    # [6] GENRE/CATEGORY MATCH SCORES
    print("  [6] Genre/Category match...")
    df = df.merge(user_aff_count, on='steamid', how='left')
    df = df.merge(user_aff_wt, on='steamid', how='left')
    df = df.merge(item_feat_mh[['appid'] + mh_cols], on='appid', how='left')
    
    genre_dot = np.zeros(N, dtype='float32')
    genre_wdot = np.zeros(N, dtype='float32')
    u_gn = np.zeros(N, dtype='float32')
    i_gn = np.zeros(N, dtype='float32')
    uw_gn = np.zeros(N, dtype='float32')
    cat_dot = np.zeros(N, dtype='float32')
    cat_wdot = np.zeros(N, dtype='float32')
    u_cn = np.zeros(N, dtype='float32')
    i_cn = np.zeros(N, dtype='float32')
    
    for col in mh_genre_cols:
        iv = df[col].fillna(0).values.astype('float32')
        uv = df[f'uaff_{col}'].fillna(0).values.astype('float32')
        uwv = df[f'uwaff_{col}'].fillna(0).values.astype('float32')
        genre_dot += uv * iv; genre_wdot += uwv * iv
        u_gn += uv**2; i_gn += iv**2; uw_gn += uwv**2
    
    for col in mh_cat_cols:
        iv = df[col].fillna(0).values.astype('float32')
        uv = df[f'uaff_{col}'].fillna(0).values.astype('float32')
        uwv = df[f'uwaff_{col}'].fillna(0).values.astype('float32')
        cat_dot += uv * iv; cat_wdot += uwv * iv
        u_cn += uv**2; i_cn += iv**2
    
    df['genre_match_cos'] = genre_dot / np.sqrt(u_gn * i_gn).clip(1e-8)
    df['genre_wmatch_cos'] = genre_wdot / np.sqrt(uw_gn * i_gn).clip(1e-8)
    df['cat_match_cos'] = cat_dot / np.sqrt(u_cn * i_cn).clip(1e-8)
    df['genre_match_dot'] = genre_dot
    df['cat_match_dot'] = cat_dot
    df['total_match_cos'] = (df['genre_match_cos'] + df['cat_match_cos']) / 2
    
    drop_aff = [c for c in df.columns if c.startswith('uaff_') or c.startswith('uwaff_')]
    df.drop(columns=drop_aff + mh_cols, inplace=True, errors='ignore')
    
    # [7] CONDITIONAL CROSS FEATURES
    print("  [7] Conditional genre features...")
    cand_genres = df[['steamid', 'appid']].merge(
        item_feat_mh[['appid'] + mh_genre_cols], on='appid', how='left'
    )
    
    cond_target_max = np.zeros(N, dtype=np.float32)
    cond_target_avg = np.zeros(N, dtype=np.float32)
    cond_pt_max = np.zeros(N, dtype=np.float32)
    cond_n_max = np.zeros(N, dtype=np.float32)
    cond_count = np.zeros(N, dtype=np.float32)  # сколько жанров матчнулось
    
    steamids_arr = df['steamid'].values
    for col in mh_genre_cols:
        genre_vals = cand_genres[col].fillna(0).values
        for i in range(N):
            if genre_vals[i] == 1:
                key = (steamids_arr[i], col)
                if key in user_genre_dict:
                    t, p, n = user_genre_dict[key]
                    if t > cond_target_max[i]:
                        cond_target_max[i] = t
                    cond_target_avg[i] += t
                    cond_count[i] += 1
                    if p > cond_pt_max[i]:
                        cond_pt_max[i] = p
                    if n > cond_n_max[i]:
                        cond_n_max[i] = n
    
    cond_target_avg = np.where(cond_count > 0, cond_target_avg / cond_count, 0)
    
    df['cond_genre_target_max'] = cond_target_max
    df['cond_genre_target_avg'] = cond_target_avg
    df['cond_genre_pt_max_log'] = np.log1p(cond_pt_max)
    df['cond_genre_n_max'] = cond_n_max
    df['cond_genre_match_count'] = cond_count
    
    del cand_genres
    
    # [8] DEV/PUB CROSS
    print("  [8] Dev/Pub cross features...")
    df_d = df[['steamid', 'appid']].merge(dev_per_appid, on='appid', how='left')
    df_d['developer'] = df_d['developer'].apply(lambda x: x if isinstance(x, list) else [])
    df_d = df_d.explode('developer')
    df_d = df_d.merge(user_dev, on=['steamid', 'developer'], how='left')
    df_d['user_dev_n_games'] = df_d['user_dev_n_games'].fillna(0)
    df_d['user_dev_total_pt'] = df_d['user_dev_total_pt'].fillna(0)
    
    da = df_d.groupby(['steamid', 'appid']).agg(
        user_dev_max_games=('user_dev_n_games', 'max'),
        user_dev_sum_games=('user_dev_n_games', 'sum'),
        user_dev_max_pt=('user_dev_total_pt', 'max'),
        user_dev_has_history=('user_dev_n_games', lambda x: int((x > 0).any())),
    ).reset_index()
    df = df.merge(da, on=['steamid', 'appid'], how='left')
    df['user_dev_max_pt_log'] = np.log1p(df['user_dev_max_pt'].fillna(0))
    
    df_p = df[['steamid', 'appid']].merge(pub_per_appid, on='appid', how='left')
    df_p['publisher'] = df_p['publisher'].apply(lambda x: x if isinstance(x, list) else [])
    df_p = df_p.explode('publisher')
    df_p = df_p.merge(user_pub, on=['steamid', 'publisher'], how='left')
    df_p['user_pub_n_games'] = df_p['user_pub_n_games'].fillna(0)
    pa = df_p.groupby(['steamid', 'appid']).agg(
        user_pub_max_games=('user_pub_n_games', 'max'),
        user_pub_has_history=('user_pub_n_games', lambda x: int((x > 0).any())),
    ).reset_index()
    df = df.merge(pa, on=['steamid', 'appid'], how='left')
    
    # User-Dev TE
    df_dt = df[['steamid', 'appid']].merge(dev_exploded, on='appid', how='left')
    df_dt = df_dt.merge(user_dev_te[['steamid', 'developer', 'user_dev_te_smooth']], 
                         on=['steamid', 'developer'], how='left')
    dt_agg = df_dt.groupby(['steamid', 'appid']).agg(
        user_dev_te_max=('user_dev_te_smooth', 'max'),
        user_dev_te_mean=('user_dev_te_smooth', 'mean'),
    ).reset_index()
    df = df.merge(dt_agg, on=['steamid', 'appid'], how='left')
    
    # [9] INTERACTION FEATURES
    print("  [9] Interaction features...")
    df['free_match'] = df['user_free_ratio'].fillna(0.5) * df['is_free'].fillna(0)
    df['als_x_centroid'] = df['als_score'] * df['centroid_sim']
    df['als_x_max_sim'] = df['als_score'] * df['max_sim_top10']
    df['als_x_dev_hist'] = df['als_score'] * df['user_dev_has_history'].fillna(0)
    df['als_dot_x_centroid'] = df['als_dot'] * df['centroid_sim']
    df['als_dot_x_max_sim'] = df['als_dot'] * df['max_sim_top10']
    df['als_x_genre_cos'] = df['als_score'] * df['genre_match_cos']
    df['als_x_total_match'] = df['als_score'] * df['total_match_cos']
    df['dev_te_x_als'] = df['item_dev_te_max'].fillna(0) * df['als_score']
    df['user_eng_x_item'] = df['user_avg_target_hist'].fillna(3) * df['item_avg_target_hist'].fillna(3)
    df['pop_vs_lib'] = df['item_n_players'].fillna(0) / df['user_total_games'].clip(lower=1)
    df['centroid_x_genre'] = df['centroid_sim'] * df['genre_match_cos']
    df['user_high_x_item_high'] = df['user_high_target_ratio'].fillna(0) * df['item_pct_high_target'].fillna(0)
    df['als_x_cond_target'] = df['als_score'] * df['cond_genre_target_max']
    df['centroid_x_cond_target'] = df['centroid_sim'] * df['cond_genre_target_max']
    df['cond_target_x_item_target'] = df['cond_genre_target_max'] * df['item_avg_target_hist'].fillna(0)
    
    # [10] CLEANUP
    print("  [10] Cleanup...")
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    df[numeric_cols] = df[numeric_cols].fillna(0)
    for c in ['loccountrycode', 'type']:
        if c in df.columns:
            df[c] = df[c].fillna('unknown')
    
    print(f"  Final: {df.shape}")
    return df


# ============================================================
# BUILD v6 — ПОЛНЫЕ 300 КАНДИДАТОВ
# ============================================================
train_v6 = assemble_v6('artifacts/train_reranker_base.parquet')
test_v6 = assemble_v6('artifacts/test_reranker_base.parquet')

train_v6.sort_values('steamid', inplace=True)
test_v6.sort_values('steamid', inplace=True)

train_v6.to_parquet('artifacts/train_v6.parquet', index=False)
test_v6.to_parquet('artifacts/test_v6.parquet', index=False)

print(f"\nTrain v6: {train_v6.shape}")
print(f"Test v6:  {test_v6.shape}")
print(f"Train pos rate: {(train_v6['target']>0).mean():.4f}")
print(f"Test pos rate:  {(test_v6['target']>0).mean():.4f}")
print(f"Candidates per user (train): {train_v6.groupby('steamid').size().mean():.0f}")


All precomputed!

Assembling v6: artifacts/train_reranker_base.parquet
  [1] ALS score features...
  [2] ALS factor products (PCA + raw)...
  [3] Item-item similarity...
    Top-K similarity (batched)...
  [4] User features...
  [5] Item features...
  [6] Genre/Category match...
  [7] Conditional genre features...
  [8] Dev/Pub cross features...
  [9] Interaction features...
  [10] Cleanup...
  Final: (1064400, 138)

Assembling v6: artifacts/test_reranker_base.parquet
  [1] ALS score features...
  [2] ALS factor products (PCA + raw)...
  [3] Item-item similarity...
    Top-K similarity (batched)...
  [4] User features...
  [5] Item features...
  [6] Genre/Category match...
  [7] Conditional genre features...
  [8] Dev/Pub cross features...
  [9] Interaction features...
  [10] Cleanup...
  Final: (266100, 138)

Train v6: (1064400, 138)
Test v6:  (266100, 138)
Train pos rate: 0.0081
Test pos rate:  0.0084
Candidates per user (train): 300


In [2]:
import pandas as pd
import numpy as np
from catboost import CatBoostRanker, Pool
from catboost.utils import get_gpu_device_count
import mlflow

print(f"GPU: {get_gpu_device_count()}")

train = pd.read_parquet("artifacts/train_v6.parquet").sort_values("steamid")
test = pd.read_parquet("artifacts/test_v6.parquet").sort_values("steamid")

drop_cols = ["steamid", "appid", "target"]
cat_features = ["loccountrycode", "type"]

for cf in cat_features:
    train[cf] = train[cf].fillna("unknown").astype(str)
    test[cf] = test[cf].fillna("unknown").astype(str)

feature_cols = [c for c in train.columns if c not in drop_cols]
print(f"Features: {len(feature_cols)}")

X_train, y_train, q_train = train[feature_cols], train["target"], train["steamid"]
X_test, y_test, q_test = test[feature_cols], test["target"], test["steamid"]

train_pool = Pool(data=X_train, label=y_train, group_id=q_train, cat_features=cat_features)
test_pool = Pool(data=X_test, label=y_test, group_id=q_test, cat_features=cat_features)

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("Steam_RecSys_v6")

results = {}

configs = [
    ("PairLogitPW_d6", {
        "loss_function": "PairLogitPairwise",
        "depth": 6, "learning_rate": 0.05,
        "border_count": 128, "bootstrap_type": "Bayesian",
    }),
    ("PairLogitPW_d7", {
        "loss_function": "PairLogitPairwise",
        "depth": 7, "learning_rate": 0.03,
        "border_count": 128, "bootstrap_type": "Bayesian",
    }),
    ("YetiRankPW", {
        "loss_function": "YetiRankPairwise",
        "depth": 7, "learning_rate": 0.03,
    }),
]

for name, extra_params in configs:
    print(f"\n{'='*60}")
    print(f"Training: {name}")
    print('='*60)
    
    with mlflow.start_run(run_name=f"v6_{name}"):
        params = {
            "iterations": 5000,
            "l2_leaf_reg": 3.0,
            "custom_metric": ["NDCG:top=10"],
            "eval_metric": "NDCG:top=10",
            "early_stopping_rounds": 200,
            "random_seed": 42,
            "task_type": "GPU",
            "devices": "0",
            "verbose": 200,
        }
        params.update(extra_params)
        mlflow.log_params(params)
        
        model = CatBoostRanker(**params)
        model.fit(train_pool, eval_set=test_pool)
        
        score = model.get_best_score()["validation"]["NDCG:top=10;type=Base"]
        best_iter = model.get_best_iteration()
        print(f"{name}: NDCG@10 = {score:.4f} (iter {best_iter})")
        
        mlflow.log_metric("best_ndcg_10", score)
        model.save_model(f"artifacts/catboost_v6_{name.lower()}.cbm")
        results[name] = (score, model, best_iter)

# ============================================================
# SUMMARY
# ============================================================
print("\n" + "="*60)
print("RESULTS (все на 300 кандидатах)")
print("="*60)
print(f"v2 baseline:        NDCG@10 = 0.3804")
print(f"v3 PairLogitPW:     NDCG@10 = 0.4121")
print(f"v4 YetiRankPW:      NDCG@10 = 0.4230")
for name, (score, _, iter_) in results.items():
    print(f"v6 {name:20s}: NDCG@10 = {score:.4f} (iter {iter_})")

# Feature importance
best_name = max(results, key=lambda x: results[x][0])
best_model = results[best_name][1]

fi = best_model.get_feature_importance(train_pool)
fi_df = pd.DataFrame({'feature': feature_cols, 'importance': fi}).sort_values('importance', ascending=False)
print(f"\nTop-50 features ({best_name}):")
print(fi_df.head(50).to_string(index=False))
fi_df.to_csv('artifacts/feature_importance_v6.csv', index=False)


GPU: 1
Features: 135


2026/05/15 23:00:13 INFO mlflow.tracking.fluent: Experiment with name 'Steam_RecSys_v6' does not exist. Creating a new experiment.



Training: PairLogitPW_d6
Groupwise loss function. OneHotMaxSize set to 10


Default metric period is 5 because NDCG is/are not implemented for GPU
Metric NDCG:top=10;type=Base is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric NDCG:top=10;type=Base is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	test: 0.2470349	best: 0.2470349 (0)	total: 72.3ms	remaining: 6m 1s
200:	test: 0.3859482	best: 0.3859482 (200)	total: 9.57s	remaining: 3m 48s
400:	test: 0.3982074	best: 0.3982074 (400)	total: 19s	remaining: 3m 37s
600:	test: 0.4029634	best: 0.4034953 (597)	total: 28.4s	remaining: 3m 27s
800:	test: 0.4028150	best: 0.4042884 (723)	total: 37.9s	remaining: 3m 18s
1000:	test: 0.4069935	best: 0.4070575 (998)	total: 47.2s	remaining: 3m 8s
1200:	test: 0.4068722	best: 0.4100644 (1158)	total: 56.7s	remaining: 2m 59s
1400:	test: 0.4105597	best: 0.4111477 (1376)	total: 1m 6s	remaining: 2m 49s
1600:	test: 0.4089681	best: 0.4127845 (1418)	total: 1m 15s	remaining: 2m 40s
bestTest = 0.4127845136
bestIteration = 1418
Shrink model to first 1419 iterations.
PairLogitPW_d6: NDCG@10 = 0.4128 (iter 1418)

Training: PairLogitPW_d7
Groupwise loss function. OneHotMaxSize set to 10


Default metric period is 5 because NDCG is/are not implemented for GPU
Metric NDCG:top=10;type=Base is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric NDCG:top=10;type=Base is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	test: 0.2759321	best: 0.2759321 (0)	total: 157ms	remaining: 13m 3s
200:	test: 0.3788694	best: 0.3791750 (199)	total: 29.6s	remaining: 11m 46s
400:	test: 0.3973645	best: 0.3988146 (395)	total: 59.1s	remaining: 11m 17s
600:	test: 0.4026432	best: 0.4027267 (598)	total: 1m 28s	remaining: 10m 47s
800:	test: 0.4020201	best: 0.4045924 (747)	total: 1m 57s	remaining: 10m 17s
1000:	test: 0.4029048	best: 0.4050548 (849)	total: 2m 27s	remaining: 9m 48s
1200:	test: 0.4083078	best: 0.4088749 (1188)	total: 2m 56s	remaining: 9m 19s
1400:	test: 0.4085182	best: 0.4091153 (1385)	total: 3m 26s	remaining: 8m 50s
1600:	test: 0.4096583	best: 0.4104833 (1496)	total: 3m 56s	remaining: 8m 21s
1800:	test: 0.4091675	best: 0.4109745 (1606)	total: 4m 25s	remaining: 7m 51s
bestTest = 0.4109745486
bestIteration = 1606
Shrink model to first 1607 iterations.
PairLogitPW_d7: NDCG@10 = 0.4110 (iter 1606)

Training: YetiRankPW
Groupwise loss function. OneHotMaxSize set to 10


Default metric period is 5 because NDCG is/are not implemented for GPU
Metric NDCG:type=Base is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric NDCG:top=10;type=Base is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric NDCG:top=10;type=Base is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	test: 0.1989218	best: 0.1989218 (0)	total: 283ms	remaining: 23m 34s
200:	test: 0.3747892	best: 0.3755337 (196)	total: 55.6s	remaining: 22m 6s
400:	test: 0.3929467	best: 0.3929467 (400)	total: 1m 50s	remaining: 21m 9s
600:	test: 0.4015282	best: 0.4019033 (589)	total: 2m 45s	remaining: 20m 14s
800:	test: 0.4069900	best: 0.4080459 (795)	total: 3m 41s	remaining: 19m 18s
1000:	test: 0.4070915	best: 0.4082898 (992)	total: 4m 36s	remaining: 18m 23s
1200:	test: 0.4077970	best: 0.4083430 (1123)	total: 5m 31s	remaining: 17m 28s
1400:	test: 0.4068451	best: 0.4088771 (1218)	total: 6m 26s	remaining: 16m 33s
bestTest = 0.408877076
bestIteration = 1218
Shrink model to first 1219 iterations.


KeyError: 'validation'